- We are given a table named famous that tracks user-follow relationships. Each record in the table represents a user_id and a follower_id, where the follower_id is following the user_id. Our task is to calculate the "famous percentage" for each user.

#### Approach
- We will get distinct users by combining the user_id and follower_id using a UNION operation.
- For each user_id, we count how many followers they have using the group by on user_id and counting the follower_id entries.
- After we have the total number of users and the count of followers for each user, we can calculate the famous percentage by dividing the number of followers by the total number of distinct users.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct, round

In [0]:
spark = SparkSession.builder.appName("FamousPercentage").getOrCreate()


In [0]:
famous_data = [
    (1, 2), (1, 3), (2, 4), (5, 1), (5, 3),
    (11, 7), (12, 8), (13, 5), (13, 10),
    (14, 12), (14, 3), (15, 14), (15, 13)
]

In [0]:
columns = ["user_id", "follower_id"]

In [0]:
df_famous = spark.createDataFrame(famous_data, columns)

In [0]:
distinct_users = df_famous.select("user_id").union(df_famous.select("follower_id")).distinct()

In [0]:
total_users = distinct_users.count()

In [0]:
followers_count = df_famous.groupBy("user_id").agg(countDistinct("follower_id").alias("follower_count"))

In [0]:
famous_percentage = followers_count.withColumn(
    "famous_percentage", 
    round((col("follower_count") / total_users), 2) * 100
)

In [0]:
famous_percentage.show()

+-------+--------------+-----------------+
|user_id|follower_count|famous_percentage|
+-------+--------------+-----------------+
|      1|             2|             15.0|
|      2|             1|              8.0|
|      5|             2|             15.0|
|     11|             1|              8.0|
|     12|             1|              8.0|
|     13|             2|             15.0|
|     14|             2|             15.0|
|     15|             2|             15.0|
+-------+--------------+-----------------+

